# 01 — Limpieza de datos

Carga de `firewall_logs.csv`, eliminación de duplicados, manejo de nulos y conversión de tipos. Documentamos cada transformación.

In [1]:
import pandas as pd

df = pd.read_csv('../datasets/firewall_logs.csv')
print('Filas originales:', len(df))
df.head()

Filas originales: 18360


,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,action,bytes,country,severity,mitre_technique
0,2024-01-15 17:43:31,45.27.52.174,10.0.0.50,42449,3389,TCP,deny,31595.0,China,critica,T1021
1,2024-02-23 04:22:38,45.45.125.58,10.0.0.2,59846,21,TCP,deny,6465.0,China,critica,T1071
2,2024-01-26 04:40:44,192.168.5.227,10.0.0.1,38357,993,TCP,allow,11410.0,México,media,NaN
3,2024-01-15 17:59:58,45.91.57.7,10.0.0.43,56687,23,ICMP,deny,22209.0,Corea del Norte,critica,T1110
4,2024-02-21 05:52:35,45.21.111.60,10.0.0.48,42620,3306,UDP,allow,45392.0,Rusia,media,T1190


## Diagnóstico inicial

In [2]:
print('Duplicados:', df.duplicated().sum())
print('\nNulos por columna:')
print(df.isna().sum())
print('\nVacíos en country:', (df['country'] == '').sum())

Duplicados: 329

Nulos por columna:
timestamp             0
src_ip                0
dst_ip                0
src_port              0
dst_port              0
protocol              0
action                0
bytes               183
country             550
severity              0
mitre_technique    8304
dtype: int64

Vacíos en country: 0


## Limpieza
1. Quitar duplicados
2. Rellenar `country` vacío con 'Desconocido'
3. Convertir `bytes` a numérico (nulos -> 0)
4. Convertir `timestamp` a datetime y derivar fecha/hora/mes

In [3]:
df = df.drop_duplicates()
df['country'] = df['country'].replace('', pd.NA).fillna('Desconocido')
df['bytes'] = pd.to_numeric(df['bytes'], errors='coerce').fillna(0).astype(int)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['fecha'] = df['timestamp'].dt.date
df['hora'] = df['timestamp'].dt.hour
df['mes'] = df['timestamp'].dt.to_period('M').astype(str)
print('Filas tras limpieza:', len(df))
df.to_csv('../datasets/firewall_clean.csv', index=False)
df.head()

Filas tras limpieza: 18031


,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,action,bytes,country,severity,mitre_technique,fecha,hora,mes
0,2024-01-15 17:43:31,45.27.52.174,10.0.0.50,42449,3389,TCP,deny,31595,China,critica,T1021,2024-01-15,17,2024-01
1,2024-02-23 04:22:38,45.45.125.58,10.0.0.2,59846,21,TCP,deny,6465,China,critica,T1071,2024-02-23,4,2024-02
2,2024-01-26 04:40:44,192.168.5.227,10.0.0.1,38357,993,TCP,allow,11410,México,media,NaN,2024-01-26,4,2024-01
3,2024-01-15 17:59:58,45.91.57.7,10.0.0.43,56687,23,ICMP,deny,22209,Corea del Norte,critica,T1110,2024-01-15,17,2024-01
4,2024-02-21 05:52:35,45.21.111.60,10.0.0.48,42620,3306,UDP,allow,45392,Rusia,media,T1190,2024-02-21,5,2024-02


## Resultado
- **329** duplicados eliminados
- **550** vacíos en `country` y **183** en `bytes` corregidos
- Dataset limpio: **18,031** filas guardadas en `firewall_clean.csv`